# X선 회절 실습

**X-ray Diffraction · XRD · 회절 패턴**

결정면 간격에 따른 회절 위치와 세기로 상과 구조를 확인하는 분석법.

소재 분야에서 이해하기: 측정 패턴을 계산 패턴과 비교해 생성된 상을 판정한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [pymatgen 문서](https://pymatgen.org/)

## 1. 브래그 법칙과 허용 반사

입방 격자의 회절 각도와 구조인자를 직접 계산합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

wavelength = 1.5406      # Cu K-alpha (angstrom)

def reflections(a, structure, max_index=4):
    """(hkl), d 간격, 2theta, 구조인자 조건을 계산합니다."""
    rows = []
    for h in range(max_index + 1):
        for k in range(max_index + 1):
            for l in range(max_index + 1):
                if h == k == l == 0:
                    continue
                if structure == 'fcc' and not (h % 2 == k % 2 == l % 2):
                    continue                              # fcc: hkl 모두 짝 또는 모두 홀
                if structure == 'bcc' and (h + k + l) % 2 != 0:
                    continue                              # bcc: h+k+l 짝수
                d = a / np.sqrt(h * h + k * k + l * l)
                ratio = wavelength / (2 * d)
                if ratio > 1:
                    continue
                rows.append((h, k, l, d, 2 * np.degrees(np.arcsin(ratio))))
    rows.sort(key=lambda row: row[4])
    return rows

for structure in ('sc', 'fcc', 'bcc'):
    rows = reflections(3.615, structure)
    print('%s 처음 5개 반사:' % structure)
    for h, k, l, d, angle in rows[:5]:
        print('   (%d%d%d) d=%.4f A  2theta=%.2f deg' % (h, k, l, d, angle))

In [ ]:
def pattern(a, structure, width=0.25):
    grid = np.linspace(20, 120, 2000)
    intensity = np.zeros_like(grid)
    for h, k, l, d, angle in reflections(a, structure):
        intensity += np.exp(-0.5 * ((grid - angle) / width) ** 2) / (h * h + k * k + l * l)
    return grid, intensity / intensity.max()

fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
for axis, structure in zip(axes, ('sc', 'fcc', 'bcc')):
    grid, intensity = pattern(3.615, structure)
    axis.plot(grid, intensity); axis.set_ylabel(structure)
axes[-1].set_xlabel('2theta (degrees)')
plt.tight_layout(); plt.show()
print('구조에 따라 사라지는 반사(소멸 조건)가 달라, 패턴만 보고 구조를 구분할 수 있습니다.')

## 2. 격자 상수를 패턴에서 되찾기

In [ ]:
measured = [row[4] for row in reflections(3.615, 'fcc')[:6]]
indices = [(row[0], row[1], row[2]) for row in reflections(3.615, 'fcc')[:6]]
estimates = []
for (h, k, l), angle in zip(indices, measured):
    d = wavelength / (2 * np.sin(np.radians(angle / 2)))
    estimates.append(d * np.sqrt(h * h + k * k + l * l))
print('반사별 격자 상수 추정 %s' % np.round(estimates, 4))
print('평균 %.4f A (참값 3.6150 A)' % np.mean(estimates))
print('\n피크 위치를 잘못 지정하거나 시료 높이가 어긋나면 계통 오차가 생깁니다.')
print('최근에는 측정 패턴에서 상을 자동 판정하는 데 딥러닝이 쓰입니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#xrd)을 여세요.